# 01 - From raw video to poses

Every downstream step - geo-referencing, tracking, the survey analytics - starts from the same two things: undistorted frames and, per frame, a *pose*: where the camera was and which way it pointed, in the metric frame the DEM lives in. This notebook shows how bambi-detection produces both, on public data, and pins down two conventions that are easy to get wrong.

It follows the [Dataset introduction](https://github.com/bambi-eco/Dataset/blob/main/introduction.ipynb) in tone; run `00_setup` first if you have not.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("."))
from _setup import ensure_environment, get_flight, find
ensure_environment()

import json
import numpy as np
import matplotlib.pyplot as plt

## Part 1 - Poses from the public dataset (no video needed)

The `base` download of a flight carries `<id>_matched_poses.json`: one entry per video frame with the drone's WGS84 position and the gimbal's angles. Reading it into arrays is the job of `bambi.io.poses` - the *file edge* of the framework. Everything after that is `bambi.geo.poses`, which speaks only numpy.

In [ ]:
from bambi.io.poses import read_poses, to_local_poses

base = get_flight("146", version="base")
pf = read_poses(find(base, "*poses*.json")[0])
print(f"schema: {pf.schema}   poses: {len(pf)}   origin: {pf.origin}")
print("lla[0]      :", pf.lla[0], "     [lat, lon, alt]")
print("rotations[0]:", pf.rotations[0], "     [tilt, roll, heading] degrees")

### One convention worth stating out loud

The public file's `pitch` field is **not** DJI gimbal pitch. DJI reports `-90` for a camera pointing straight down; the dataset stores the *pose* convention - tilt from nadir - so a nadir flight reads `~0` (kept unwrapped as `360.0`). The Dataset repository's own converter copies `[pitch, roll, yaw]` into `rotation` verbatim, and `read_poses` does the same, exposing it as `rotations` and wrapping to `[0, 360)`.

Adding 90 again here would tilt every camera to the horizon. The framework keeps the two sources apart: DJI SRT / AirData angles go through `gimbal_to_rotation` (which does add 90); the public dataset does not.

In [ ]:
tilt = pf.rotations[:, 0]
print(f"tilt on this flight: {tilt.min():.2f} .. {tilt.max():.2f} deg  -> a nadir flight, as it should be")

### Into the DEM-local frame

The pipeline computes in metres relative to the DEM's origin, on the axes of a projected CRS. `to_local_poses` does that conversion for either schema; for a geographic file it needs the CRS the DEM will use (UTM 33N here).

In [ ]:
poses = to_local_poses(pf, epsg=32633)
P, R = poses.positions, poses.rotations
print("positions", P.shape, " rotations", R.shape)
print(f"east   {P[:,0].min():8.1f} .. {P[:,0].max():8.1f} m")
print(f"north  {P[:,1].min():8.1f} .. {P[:,1].max():8.1f} m")
print(f"up     {P[:,2].min():8.1f} .. {P[:,2].max():8.1f} m   (above the origin altitude {poses.origin.altitude:.1f} m)")

fig, ax = plt.subplots(figsize=(6.5, 5))
sc = ax.scatter(P[:, 0], P[:, 1], c=P[:, 2], s=3, cmap="viridis")
ax.set_aspect("equal"); ax.set_xlabel("east (m)"); ax.set_ylabel("north (m)")
ax.set_title("flight 146 in the DEM-local frame"); plt.colorbar(sc, label="up (m)"); plt.show()

### The maths behind it, exposed

`bambi.geo.poses` is deliberately small and array-shaped, so you can call it on anything - a pose you typed, a batch from a tracker, a test fixture. Round trips are exact.

In [ ]:
from bambi.geo.poses import geographic_to_local, local_to_geographic, gimbal_to_rotation, rotation_to_gimbal, make_origin

o = make_origin(46.6448, 14.5593, 450.0, epsg=32633)
one = geographic_to_local([46.6451, 14.5605, 479.5], o)
print("one pose  ->", one.round(3))
print("and back  ->", local_to_geographic(one, o).round(6))

# DJI-style gimbal angles (SRT / AirData): pitch -90 = nadir
print("gimbal [-90,0,51.6] -> pose", gimbal_to_rotation([-90, 0, 51.6]))
print("gimbal [  0,0,51.6] -> pose", gimbal_to_rotation([0, 0, 51.6]), " (horizon)")
print("round trip:", rotation_to_gimbal(gimbal_to_rotation([-45, 0, 300])))

## Part 2 - Extracting frames from a raw flight

The `raw` download is the original DJI recording: MP4 + SRT + AirData log + the camera calibrations. Flight `258` is the smallest one (196 MB), an M3T thermal/RGB pair, so it is the one used here and in CI.

In [ ]:
raw = get_flight("258", version="raw")
for p in sorted(raw.iterdir()):
    if p.is_file():
        print(f"{p.stat().st_size/1e6:8.1f} MB  {p.name}")

### First: does the calibration belong to this camera?

This check exists because of a real incident. A calibration is a camera matrix + distortion; OpenCV will apply *any* calibration to *any* video without complaint. Apply one made for a 640x512 sensor to a 1280x1024 stream and the frames come out re-centred on the wrong pixel and zoomed 2x - every detection made on them points tens of degrees off. Nothing downstream can notice.

The tell is cheap: a calibration's principal point sits near the centre of the image it was made on, so `2*cx, 2*cy` recovers that size and can be compared with the media. Here the pairing is right, and we can show what the wrong one would have looked like.

In [ ]:
from bambi.io.calibration import load_calibration, media_resolution
from bambi.geo.calibration import check_resolution, implied_resolution

thermal_video = find(raw, "*_T_*.MP4")[0]
mtx, dist = load_calibration(raw / "T_calib.json")
w, h = media_resolution([thermal_video])
print(f"thermal video is {w}x{h}; calibration implies {tuple(round(v) for v in implied_resolution(mtx))}")

ok = check_resolution(mtx, w, h)
print(f"this pairing                -> {ok.severity!r}   (deviation {ok.deviation:.1%})")

# The lion-flight mistake: this very calibration (M3T, 640x512) applied to an M30T 1280x1024 stream.
bad = check_resolution(mtx, 1280, 1024)
print(f"same calibration @1280x1024 -> {bad.severity!r} (deviation {bad.deviation:.1%}, focal length scaled by {bad.scale:.2f}x)")

### The field of view the extractor will write

The extractors square the frames and force equal fx/fy; the resulting vertical field of view is what every projection downstream assumes. `fovy_after_undistortion` reproduces that recipe, so it can be predicted - and, below, checked against what the extractor actually writes.

In [ ]:
from bambi.geo.calibration import fovy_after_undistortion

side = min(w, h)
predicted_fovy = fovy_after_undistortion(mtx, dist, (w, h), (side, side))
print(f"predicted fovy of the extracted {side}x{side} frames: {predicted_fovy:.4f} deg")

### Extract a slice

`TimedPoseExtractor` reads the video, undistorts each frame with the calibration, matches SRT timestamps to the AirData log (interpolating the log to frame time), and writes frames + `poses.json`. This recording starts with the drone hovering and the gimbal on the horizon; the SRT shows it swinging down to nadir at frame 89, so we `skip` to there and take a short slice. The same call without `skip`/`limit` processes the whole flight.

In [ ]:
from pyproj import CRS, Transformer
from dateutil import tz
from bambi.airdata.air_data_frame import AirDataFrame
from bambi.video.calibrated_video_frame_accessor import CalibratedVideoFrameAccessor
from bambi.webgl.timed_pose_extractor import TimedPoseExtractor
from bambi.domain.camera import Camera

out = raw / "frames_t_demo"; out.mkdir(exist_ok=True)
origin = AirDataFrame(); origin.latitude, origin.longitude, origin.altitude = 46.9195, 15.7314, 480.0

extractor = TimedPoseExtractor(
    CalibratedVideoFrameAccessor(json.load(open(raw / "T_calib.json")), preserve_aspect_ratio=False),
    rel_transformer=Transformer.from_crs(CRS.from_epsg(4326), CRS.from_epsg(32633)),
    camera_name=Camera.from_string("T"), use_gimbal_heading=False)
extractor.extract(str(out), str(raw / "air_data.csv"), [str(thermal_video)],
                  [str(find(raw, "*_T_*.SRT")[0])], origin=origin, include_gps=True,
                  skip=89, limit=300, sampling_rate=0, timezone=tz.tzoffset(None, 2 * 3600))

written = read_poses(out / "poses.json")
frames = sorted(out.glob("*.jpg"))
print(f"{len(written)} poses written, {len(frames)} frames")
print(f"fovy written by the extractor: {written.fovy[0]:.4f}   predicted above: {predicted_fovy:.4f}")

### What came out

The extractor's output is the *pipeline* schema - already DEM-local `location`/`rotation` - so it reads straight into the same arrays as Part 1. The tilt column shows the gimbal settling onto nadir (DJI `gb_pitch` -88 -> -90 becomes tilt 2 -> 0).

In [ ]:
ex = to_local_poses(written, epsg=32633)
print("first rotations [tilt, roll, heading]:")
print(np.round(ex.rotations[:5], 2))
img = plt.imread(frames[len(frames) // 2])
plt.figure(figsize=(5, 5)); plt.imshow(img, cmap="gray"); plt.axis("off")
plt.title(f"an undistorted, squared thermal frame ({img.shape[1]}x{img.shape[0]})"); plt.show()

## Where this goes next

`02_georeference` takes these poses and a DEM and puts detections on the ground - and, because every public flight is nadir, adds a synthetic oblique scene to show what the pointing convention has to get right.